<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Regression metrics

**[Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/)** · Machine Learning: From Problem to Reliable Model · Module 3, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure the errors of a forecast that predicts a number, with the mean error, MAE, RMSE, MAPE and R², and to choose the metric that matches what a wrong forecast costs.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | What regression is, from [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/). Why we evaluate on data that a forecast did not learn from, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/). |
| **You do not need** | The local project. The setup below downloads the data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M03-L02-regression-metrics/regression-metrics-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Five days, two forecasts

A regression model predicts a number. The **error** of one prediction is the real value minus the predicted value. A positive error means the forecast was too low; a negative error means it was too high.

Here are five days, from Thursday to Monday. **Forecast A** says 30 on a weekday and 15 at the weekend. **Forecast B** says 26 every day.

- The **mean error** (or **bias**) is the mean of the errors with their signs. It shows the direction: too low or too high on average.
- The **MAE** (mean absolute error) drops the signs first. It is the typical size of a miss, in tickets.
- The **RMSE** (root mean squared error) squares each error, takes the mean, then the square root. Large misses count much more.

> **Predict.** Forecast A misses by 10 on Monday and by 2 or less on the other days. Which will be larger, its MAE or its RMSE? Write your prediction, then run the cell.

In [ ]:
real = np.array([31, 30, 14, 13, 40])
forecast_a = np.array([30, 30, 15, 15, 30])
forecast_b = np.full(5, 26)
errors = real - forecast_a
print("errors of A:", errors)
print("mean error", errors.mean(), "| MAE", np.abs(errors).mean(), "| RMSE", round(float(np.sqrt((errors ** 2).mean())), 2))

> **Check.** You should see `errors of A: [ 1  0 -1 -2 10]` and `mean error 1.6 | MAE 2.8 | RMSE 4.6`.

By hand: MAE = (1 + 0 + 1 + 2 + 10) ÷ 5 = 2.8. The squares are 1, 0, 1, 4 and 100, so RMSE = √(106 ÷ 5) = √21.2 = 4.6. Monday gives 100 of the 106. **RMSE is much larger than MAE when a few misses are large.**

Now forecast B. Its errors are +5, +4, −12, −13 and +14.

> **Predict.** What is the mean error of B? Is B a better forecast than A? Then run the cell.

In [ ]:
errors_b = real - forecast_b
print("errors of B:", errors_b)
print("mean error", errors_b.mean(), "| MAE", np.abs(errors_b).mean())

> **Check.** You should see `mean error -0.4 | MAE 9.6`.

The mean error of B is almost 0, because its large positive and negative errors **cancel**. But B misses by about 10 tickets on a typical day. So the mean error shows the direction, never the size. Always read it next to MAE or RMSE.

## 2. MAPE: the miss as a percentage

A miss of 5 tickets is 42% of a quiet day with 12 tickets, but only 12.5% of a busy day with 40. The **MAPE** (mean absolute percentage error) divides each absolute error by the real value, then takes the mean, as a percentage.

> **Your turn.** Compute the MAPE of forecast A with numpy: `np.abs(real - forecast_a) / real` gives each share. Take the mean, multiply by 100, and round to 1 decimal. Put the result in `mape_a`, as a Python number (`float(...)`). Then run the check cell.

In [ ]:
mape_a = round(float(100 * np.mean(np.abs(real - forecast_a) / real)), 1)
print(mape_a)

In [ ]:
expect('mape_a', mape_a, 10.2)

The five shares are 3.2%, 0%, 7.1%, 15.4% and 25.0%. MAPE breaks **near zero**: a day with 2 real tickets and a forecast of 4 is a 100% error, and a day with 0 tickets cannot be divided at all.

## 3. R², and all five metrics in one function

**R²** compares the forecast with a very simple one: the mean of the real values (here 25.6). R² = 1 is perfect, R² = 0 is no better than the mean, and R² below 0 is worse than the mean.

The next cell defines `regression_report()`, which returns the five metrics of a forecast. scikit-learn computes MAE, the mean squared error (its square root is RMSE), MAPE (as a share, so the function multiplies by 100) and R².

> **Predict.** Forecast B says 26 every day, close to the real mean of 25.6. What R² do you expect for B? Then run the cell.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

def regression_report(real, forecast):
    """Five numbers that describe the errors of a forecast."""
    errors = real - forecast
    return {"mean error": round(float(errors.mean()), 2),
            "MAE": round(mean_absolute_error(real, forecast), 2),
            "RMSE": round(mean_squared_error(real, forecast) ** 0.5, 2),
            "MAPE %": round(100 * mean_absolute_percentage_error(real, forecast), 1),
            "R2": round(r2_score(real, forecast), 3)}

print("A:", regression_report(real, forecast_a))
print("B:", regression_report(real, forecast_b))

> **Check.** You should see:
>
> - `A: {'mean error': 1.6, 'MAE': 2.8, 'RMSE': 4.6, 'MAPE %': 10.2, 'R2': 0.807}`
> - `B: {'mean error': -0.4, 'MAE': 9.6, 'RMSE': 10.49, 'MAPE %': 50.0, 'R2': -0.001}`

B's R² is about 0: it is no better than the mean, because it almost **is** the mean.

## 4. Larkfield's tickets per day: three forecasts

No model exists yet in this course, and a forecast does not need one. Each forecast below learns only from the days before 1 May 2026 (the **training days**), and is evaluated on the 61 days of May and June 2026. Run the cell.

In [ ]:
daily = pd.read_csv("data/daily_tickets.csv", parse_dates=["date"])
train_days = daily[daily["date"] < "2026-05-01"]
eval_days = daily[daily["date"] >= "2026-05-01"].copy()
print(len(train_days), "training days,", len(eval_days), "days to evaluate")
print("mean per day: training", round(train_days["tickets"].mean(), 2), "| May-June", round(eval_days["tickets"].mean(), 2))

> **Check.** You should see `669 training days, 61 days to evaluate` and `mean per day: training 25.32 | May-June 37.59`.

May and June are much busier: the garden season, and a shop that grows. The next cell makes three forecasts:

1. **train mean:** the mean of all training days;
2. **weekday or weekend mean:** one mean for weekdays, one for weekends;
3. **same weekday and month:** the mean of the training days with the same weekday in the same month (for a Monday in May: the Mondays of May 2025).

Run the cell.

In [ ]:
eval_days["train mean"] = train_days["tickets"].mean()
weekend_means = train_days.groupby("is_weekend")["tickets"].mean()
eval_days["weekday or weekend mean"] = eval_days["is_weekend"].map(weekend_means)
same_means = train_days.groupby(["month", "weekday"])["tickets"].mean()
eval_days["same weekday and month"] = [same_means[(month, weekday)]
                                       for month, weekday in zip(eval_days["month"], eval_days["weekday"])]
print(weekend_means.round(2).to_dict())

> **Check.** You should see `{0: 29.99, 1: 13.54}`: the weekday mean and the weekend mean.

> **Predict.** All three forecasts learned from a quieter time. Will their mean errors be positive (too low) or negative (too high)? Which forecast will have the lowest MAE? Then run the cell.

In [ ]:
forecasts = ["train mean", "weekday or weekend mean", "same weekday and month"]
print(pd.DataFrame({name: regression_report(eval_days["tickets"], eval_days[name]) for name in forecasts}).T)

> **Check.** You should see three rows:
>
> - train mean: mean error 12.27, MAE 14.95, RMSE 17.27, MAPE 38.3, R² −1.018;
> - weekday or weekend mean: 12.45, 12.54, 14.08, 32.2, −0.340;
> - same weekday and month: 5.00, 6.13, 7.83, 16.4, 0.586.

All three are too low on average. The first two have R² **below zero**: they are worse than the mean of May and June, which they could not know. "Same weekday and month" is the best: about 6 tickets off on a typical day.

The chart shows the real counts and the three forecasts. Run the cell.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(eval_days["date"], eval_days["tickets"], "o-", label="real tickets")
for name in forecasts:
    ax.plot(eval_days["date"], eval_days[name], label=name)
ax.set_xlabel("Day")
ax.set_ylabel("Tickets per day")
ax.legend()
plt.show()

> **Check.** You should see the real counts go up and down every week, mostly between 36 and 52 on weekdays and 15 and 26 at weekends. The train mean is a flat line at about 25. The other two forecasts follow the weekly shape, but are below the real counts on most days.

## 5. Change one thing: the last four weeks

The forecasts are too low because the shop grows. A forecast that uses only **recent** days may follow the growth.

> **Predict.** A forecast uses the mean of the same weekday over the last 4 weeks before May (3 to 30 April 2026). Will its mean error be closer to 0 than 5.00? Will its MAE be lower than 6.13?

> **Your turn.** Make `last_4_weeks`: the training days from `"2026-04-03"`. Compute the mean tickets per weekday in it, map them onto `eval_days["weekday"]`, and put the forecast in the column `"same weekday, last 4 weeks"`. Then put its `regression_report()` in `report_4_weeks`, and run the check cell.

In [ ]:
last_4_weeks = train_days[train_days["date"] >= "2026-04-03"]
weekday_means = last_4_weeks.groupby("weekday")["tickets"].mean()
eval_days["same weekday, last 4 weeks"] = eval_days["weekday"].map(weekday_means)
report_4_weeks = regression_report(eval_days["tickets"], eval_days["same weekday, last 4 weeks"])
print(report_4_weeks)

In [ ]:
expect_hash('report_4_weeks', report_4_weeks, '49e77fd74321fcd3')

Recent data reduces the bias a lot, because April was already busy. The cost: a forecast from only 4 weeks moves with every unusual week (a holiday, an outage), because it averages few days.

## 6. Failure case: a small MAE that hides a bias

Tomás compares the forecasts by MAE only, and says "same weekday and month is off by 6.13 tickets a day, good enough". Omar plans the May shifts with it, and the queue is long almost every evening.

> **Predict.** On how many of the 61 days is "same weekday and month" too low? Then run the cell.

In [ ]:
errors = eval_days["tickets"] - eval_days["same weekday and month"]
print("mean error:", round(errors.mean(), 2), "| days too low:", (errors > 0).sum(), "of", len(errors))

> **Check.** You should see `mean error: 5.0 | days too low: 50 of 61`.

MAE measures the size of the misses, not their direction. A forecast that is too low almost every day leaves the team short every day. **Report the mean error next to MAE or RMSE, every time**, and find the cause of a large bias before you trust the forecast.

## Next

You can now measure a forecast of a number in five ways, and choose the metric from what a wrong forecast costs: for Larkfield's shifts, RMSE with MAE and the mean error. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/) has the decision table of real cases and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.